In [1]:
import os
import json
import random
import time
import tracemalloc
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.stats
from itertools import combinations, product
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, mean_squared_error,
    balanced_accuracy_score, average_precision_score, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from statsmodels.stats.multitest import multipletests
from scipy.stats import friedmanchisquare, wilcoxon

warnings.filterwarnings('ignore')

GLOBAL_SEED   = 42
LAMBDA_REG    = 1e-5
N_SPLITS      = 10
LEARNING_RATE = 0.005
N_EPOCHS      = 300
N_CLASSES     = 2
ACTIVATIONS   = ['linear', 'sigmoid', 'relu', 'tanh']
HIDDEN_SIZES  = [50, 100, 200]

DATASET_NAME = 'Breast Cancer Wisconsin Diagnostic (UCI id=17)'
DATASET_ID   = 17
OUTPUT_JSON  = 'results_BreastCancer.json'

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_all_seeds(GLOBAL_SEED)

raw_dataset  = fetch_ucirepo(id=DATASET_ID)
X_raw        = raw_dataset.data.features
y_raw        = raw_dataset.data.targets.values.flatten()
y            = np.where(y_raw == 'M', 1, 0).astype(np.int64)

stratified_kfold = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=GLOBAL_SEED)
fold_indices     = [(train.tolist(), val.tolist()) for train, val in stratified_kfold.split(np.zeros(len(y)), y)]

class_distribution  = {'benign (0)': int(np.sum(y == 0)), 'malignant (1)': int(np.sum(y == 1))}
n_features_original = int(X_raw.shape[1])
n_samples           = int(len(y))

print(f'Dataset  : {DATASET_NAME}')
print(f'Samples  : {n_samples}  Classes: {class_distribution}')
print(f'Features : {n_features_original}')
print(f'Folds    : {N_SPLITS}  Seed: {GLOBAL_SEED}')

Dataset  : Breast Cancer Wisconsin Diagnostic (UCI id=17)
Samples  : 569  Classes: {'benign (0)': 357, 'malignant (1)': 212}
Features : 30
Folds    : 10  Seed: 42


In [2]:
class ELM(nn.Module):
    def __init__(self, n_features, n_hidden, n_classes, activation='sigmoid', seed=GLOBAL_SEED):
        super().__init__()
        torch.manual_seed(seed); np.random.seed(seed)
        self.activation = activation
        self.W    = nn.Parameter(torch.randn(n_features, n_hidden), requires_grad=False)
        self.b    = nn.Parameter(torch.zeros(1, n_hidden),          requires_grad=False)
        self.beta = nn.Parameter(torch.zeros(n_hidden, n_classes),  requires_grad=True)
    def activate(self, x):
        if self.activation == 'linear':  return x
        if self.activation == 'sigmoid': return torch.sigmoid(x)
        if self.activation == 'relu':    return F.relu(x)
        if self.activation == 'tanh':    return torch.tanh(x)
        return torch.sigmoid(x)
    def forward(self, x, return_hidden=False):
        hidden = self.activate(torch.matmul(x, self.W) + self.b)
        logits = torch.matmul(hidden, self.beta)
        if return_hidden: return logits, hidden
        return F.softmax(logits, dim=1).clamp(min=1e-7, max=1.0 - 1e-7)

class ELM_SelfGate(nn.Module):
    def __init__(self, n_features, n_hidden, n_classes, activation='sigmoid', seed=GLOBAL_SEED):
        super().__init__()
        torch.manual_seed(seed); np.random.seed(seed)
        self.activation = activation
        self.W    = nn.Parameter(torch.randn(n_features, n_hidden), requires_grad=False)
        self.b    = nn.Parameter(torch.zeros(1, n_hidden),          requires_grad=False)
        self.beta = nn.Parameter(torch.zeros(n_hidden, n_classes),  requires_grad=True)
    def activate(self, x):
        if self.activation == 'linear':  return x
        if self.activation == 'sigmoid': return torch.sigmoid(x)
        if self.activation == 'relu':    return F.relu(x)
        if self.activation == 'tanh':    return torch.tanh(x)
        return torch.sigmoid(x)
    def forward(self, x, return_hidden=False):
        pre_gate = self.activate(torch.matmul(x, self.W) + self.b)
        hidden   = pre_gate * pre_gate
        logits   = torch.matmul(hidden, self.beta)
        if return_hidden: return logits, hidden
        return F.softmax(logits, dim=1).clamp(min=1e-7, max=1.0 - 1e-7)

class ELM_DNM(nn.Module):
    def __init__(self, n_features, n_hidden, n_classes, act_syn='sigmoid', act_dend='sigmoid', seed=GLOBAL_SEED):
        super().__init__()
        torch.manual_seed(seed); np.random.seed(seed)
        self.act_syn = act_syn; self.act_dend = act_dend
        self.W_syn  = nn.Parameter(torch.randn(n_features, n_hidden), requires_grad=False)
        self.b_syn  = nn.Parameter(torch.zeros(1, n_hidden),          requires_grad=False)
        self.W_dend = nn.Parameter(torch.randn(n_hidden, n_hidden),   requires_grad=False)
        self.b_dend = nn.Parameter(torch.zeros(1, n_hidden),          requires_grad=False)
        self.beta   = nn.Parameter(torch.zeros(n_hidden, n_classes),  requires_grad=True)
    def activate(self, x, kind):
        if kind == 'linear':  return x
        if kind == 'sigmoid': return torch.sigmoid(x)
        if kind == 'relu':    return F.relu(x)
        if kind == 'tanh':    return torch.tanh(x)
        return torch.sigmoid(x)
    def forward(self, x, return_hidden=False):
        synaptic  = self.activate(torch.matmul(x, self.W_syn) + self.b_syn, self.act_syn)
        dendritic = self.activate(torch.matmul(synaptic, self.W_dend) + self.b_dend, self.act_dend)
        hidden    = synaptic * dendritic
        logits    = torch.matmul(hidden, self.beta)
        if return_hidden: return logits, hidden
        return F.softmax(logits, dim=1).clamp(min=1e-7, max=1.0 - 1e-7)

class StandardDNM(nn.Module):
    def __init__(self, n_features, n_hidden, n_classes, act_syn='sigmoid', act_dend='sigmoid', seed=GLOBAL_SEED):
        super().__init__()
        torch.manual_seed(seed); np.random.seed(seed)
        self.act_syn = act_syn; self.act_dend = act_dend
        self.W_syn  = nn.Parameter(torch.randn(n_features, n_hidden))
        self.b_syn  = nn.Parameter(torch.zeros(1, n_hidden))
        self.W_dend = nn.Parameter(torch.randn(n_hidden, n_hidden))
        self.b_dend = nn.Parameter(torch.zeros(1, n_hidden))
        self.beta   = nn.Parameter(torch.randn(n_hidden, n_classes))
    def activate(self, x, kind):
        if kind == 'linear':  return x
        if kind == 'sigmoid': return torch.sigmoid(x)
        if kind == 'relu':    return F.relu(x)
        if kind == 'tanh':    return torch.tanh(x)
        return torch.sigmoid(x)
    def forward(self, x):
        synaptic  = self.activate(torch.matmul(x, self.W_syn) + self.b_syn, self.act_syn)
        dendritic = self.activate(torch.matmul(synaptic, self.W_dend) + self.b_dend, self.act_dend)
        return F.softmax(torch.matmul(synaptic * dendritic, self.beta), dim=1).clamp(min=1e-7, max=1.0 - 1e-7)

class MODN(nn.Module):
    def __init__(self, n_features, n_hidden, n_classes, act_syn='sigmoid', act_dend='sigmoid', seed=GLOBAL_SEED):
        super().__init__()
        torch.manual_seed(seed); np.random.seed(seed)
        self.n_hidden = n_hidden; self.n_classes = n_classes
        self.act_syn = act_syn; self.act_dend = act_dend
        self.W_syn   = nn.Parameter(torch.randn(n_features, n_hidden))
        self.b_syn   = nn.Parameter(torch.zeros(1, n_hidden))
        self.W_dend  = nn.Parameter(torch.randn(n_hidden, n_hidden))
        self.b_dend  = nn.Parameter(torch.zeros(1, n_hidden))
        self.W_proj  = nn.Parameter(torch.randn(n_hidden, n_hidden * n_classes))
        self.b_proj  = nn.Parameter(torch.zeros(1, n_hidden * n_classes))
        self.W_final = nn.Parameter(torch.randn(n_hidden, n_classes))
    def activate(self, x, kind):
        if kind == 'linear':  return x
        if kind == 'sigmoid': return torch.sigmoid(x)
        if kind == 'relu':    return F.relu(x)
        if kind == 'tanh':    return torch.tanh(x)
        return torch.sigmoid(x)
    def forward(self, x):
        synaptic  = self.activate(torch.matmul(x, self.W_syn) + self.b_syn, self.act_syn)
        dendritic = self.activate(torch.matmul(synaptic, self.W_dend) + self.b_dend, self.act_dend)
        projected = (torch.matmul(synaptic * dendritic, self.W_proj) + self.b_proj).view(-1, self.n_classes, self.n_hidden)
        return F.softmax(torch.matmul(projected, self.W_final).mean(dim=1), dim=1).clamp(min=1e-7, max=1.0 - 1e-7)

print('Model classes ready.')

Model classes ready.


In [3]:
def elm_solve(hidden_matrix, target_matrix, regularization):
    if regularization > 0.0:
        reg_matrix  = (regularization ** 0.5) * torch.eye(hidden_matrix.shape[1], dtype=hidden_matrix.dtype)
        augmented_H = torch.cat([hidden_matrix, reg_matrix], dim=0)
        augmented_Y = torch.cat([target_matrix, torch.zeros(hidden_matrix.shape[1], target_matrix.shape[1], dtype=target_matrix.dtype)], dim=0)
    else:
        augmented_H = hidden_matrix
        augmented_Y = target_matrix
    return torch.linalg.lstsq(augmented_H, augmented_Y, driver='gelsd').solution

def compute_metrics(y_true, y_pred, positive_class_probs, full_probs):
    conf_matrix = confusion_matrix(y_true, y_pred)
    if conf_matrix.shape == (2, 2):
        tn, fp, fn, tp = conf_matrix.ravel()
    else:
        tn, fp, fn, tp = 0, 0, 0, 0
    sensitivity = float(tp) / float(tp + fn) if (tp + fn) > 0 else 0.0
    specificity = float(tn) / float(tn + fp) if (tn + fp) > 0 else 0.0
    return {
        'accuracy':          float(accuracy_score(y_true, y_pred)),
        'precision':         float(precision_score(y_true, y_pred, average='weighted', zero_division=0)),
        'recall':            float(recall_score(y_true, y_pred, average='weighted', zero_division=0)),
        'f1':                float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'sensitivity':       sensitivity,
        'specificity':       specificity,
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'roc_auc':           float(roc_auc_score(y_true, positive_class_probs)),
        'pr_auc':            float(average_precision_score(y_true, positive_class_probs)),
        'log_loss':          float(log_loss(y_true, full_probs)),
        'rmse':              float(np.sqrt(mean_squared_error(y_true, positive_class_probs))),
        'confusion_matrix':  [[int(v) for v in row] for row in conf_matrix.tolist()]
    }

def train_elm_analytical(model, X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor, regularization=LAMBDA_REG):
    model.eval(); tracemalloc.start(); t_start = time.perf_counter()
    with torch.no_grad(): _, hidden_matrix = model(X_train_tensor, return_hidden=True)
    target_matrix = F.one_hot(y_train_tensor, N_CLASSES).float()
    model.beta.data = elm_solve(hidden_matrix, target_matrix, regularization)
    train_time = time.perf_counter() - t_start
    peak_memory, _ = tracemalloc.get_traced_memory(); tracemalloc.stop()
    t_infer = time.perf_counter()
    with torch.no_grad(): predicted_probs = model(X_val_tensor)
    inference_time = time.perf_counter() - t_infer
    y_true = y_val_tensor.cpu().numpy(); y_pred = torch.argmax(predicted_probs, dim=1).cpu().numpy()
    positive_probs = predicted_probs.cpu().numpy()[:, 1]; full_probs_np = predicted_probs.cpu().numpy()
    result = compute_metrics(y_true, y_pred, positive_probs, full_probs_np)
    result['train_time'] = float(train_time); result['time_per_epoch'] = float('nan')
    result['inference_time'] = float(inference_time); result['memory_mb'] = float(peak_memory) / 1e6
    return result

def train_elm_no_regularization(model, X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor):
    return train_elm_analytical(model, X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor, regularization=0.0)

def train_gradient_descent(model, X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor):
    model.train(); criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    tracemalloc.start(); t_start = time.perf_counter()
    for _ in range(N_EPOCHS):
        optimizer.zero_grad()
        loss = criterion(model(X_train_tensor), y_train_tensor)
        loss.backward(); optimizer.step()
    train_time = time.perf_counter() - t_start; time_per_epoch = train_time / N_EPOCHS
    peak_memory, _ = tracemalloc.get_traced_memory(); tracemalloc.stop()
    model.eval(); t_infer = time.perf_counter()
    with torch.no_grad(): predicted_probs = model(X_val_tensor)
    inference_time = time.perf_counter() - t_infer
    y_true = y_val_tensor.cpu().numpy(); y_pred = torch.argmax(predicted_probs, dim=1).cpu().numpy()
    positive_probs = predicted_probs.cpu().numpy()[:, 1]; full_probs_np = predicted_probs.cpu().numpy()
    result = compute_metrics(y_true, y_pred, positive_probs, full_probs_np)
    result['train_time'] = float(train_time); result['time_per_epoch'] = float(time_per_epoch)
    result['inference_time'] = float(inference_time); result['memory_mb'] = float(peak_memory) / 1e6
    return result

def train_sklearn_classifier(classifier_factory, X_train_np, y_train_np, X_val_np, y_val_np):
    tracemalloc.start(); t_start = time.perf_counter()
    classifier = classifier_factory(); classifier.fit(X_train_np, y_train_np)
    train_time = time.perf_counter() - t_start
    peak_memory, _ = tracemalloc.get_traced_memory(); tracemalloc.stop()
    t_infer = time.perf_counter(); y_pred = classifier.predict(X_val_np)
    full_probs_np = classifier.predict_proba(X_val_np); inference_time = time.perf_counter() - t_infer
    positive_probs = full_probs_np[:, 1]
    result = compute_metrics(y_val_np, y_pred, positive_probs, full_probs_np)
    result['train_time'] = float(train_time); result['time_per_epoch'] = float('nan')
    result['inference_time'] = float(inference_time); result['memory_mb'] = float(peak_memory) / 1e6
    return result

print('Training functions ready.')

Training functions ready.


In [4]:
def build_model_registry(n_features):
    registry = []
    for activation, n_hidden in product(ACTIVATIONS, HIDDEN_SIZES):
        registry.append((f'ELM-{activation}-h{n_hidden}', lambda nf=n_features, act=activation, nh=n_hidden: ELM(nf, nh, N_CLASSES, act), train_elm_analytical, 'analytical'))
    for activation, n_hidden in product(ACTIVATIONS, HIDDEN_SIZES):
        registry.append((f'ELM-SelfGate-{activation}-h{n_hidden}', lambda nf=n_features, act=activation, nh=n_hidden: ELM_SelfGate(nf, nh, N_CLASSES, act), train_elm_analytical, 'analytical'))
    for activation, n_hidden in product(ACTIVATIONS, HIDDEN_SIZES):
        registry.append((f'ELM-DNM-noreg-{activation}-h{n_hidden}', lambda nf=n_features, act=activation, nh=n_hidden: ELM_DNM(nf, nh, N_CLASSES, act, act), train_elm_no_regularization, 'analytical'))
    for activation, n_hidden in product(ACTIVATIONS, HIDDEN_SIZES):
        registry.append((f'ELM-DNM-{activation}-h{n_hidden}', lambda nf=n_features, act=activation, nh=n_hidden: ELM_DNM(nf, nh, N_CLASSES, act, act), train_elm_analytical, 'analytical'))
    for activation, n_hidden in product(ACTIVATIONS, HIDDEN_SIZES):
        registry.append((f'DNM-{activation}-h{n_hidden}', lambda nf=n_features, act=activation, nh=n_hidden: StandardDNM(nf, nh, N_CLASSES, act, act), train_gradient_descent, 'gradient'))
    for activation, n_hidden in product(ACTIVATIONS, HIDDEN_SIZES):
        registry.append((f'MODN-{activation}-h{n_hidden}', lambda nf=n_features, act=activation, nh=n_hidden: MODN(nf, nh, N_CLASSES, act, act), train_gradient_descent, 'gradient'))
    registry.append(('RandomForest', lambda: RandomForestClassifier(n_estimators=100, random_state=GLOBAL_SEED), None, 'sklearn'))
    registry.append(('SVM', lambda: SVC(probability=True, random_state=GLOBAL_SEED), None, 'sklearn'))
    registry.append(('MLP', lambda: MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=GLOBAL_SEED), None, 'sklearn'))
    return registry

results_all = {
    'metadata': {
        'dataset': DATASET_NAME, 'dataset_id': DATASET_ID,
        'n_samples': n_samples, 'n_features_original': n_features_original,
        'class_distribution': class_distribution, 'n_folds': N_SPLITS,
        'random_state': GLOBAL_SEED, 'activations_tested': ACTIVATIONS,
        'hidden_sizes_tested': HIDDEN_SIZES, 'regularization_lambda': LAMBDA_REG,
        'gradient_learning_rate': LEARNING_RATE, 'gradient_epochs': N_EPOCHS,
        'fold_indices': fold_indices, 'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
    },
    'models': {}
}
sample_registry = build_model_registry(10)
print(f'Models per fold  : {len(sample_registry)}')
print(f'Total evaluations: {len(sample_registry) * N_SPLITS}')

Models per fold  : 75
Total evaluations: 750


In [5]:
for fold_idx, (train_indices, val_indices) in enumerate(fold_indices):
    set_all_seeds(GLOBAL_SEED + fold_idx)
    t_preprocess_start = time.perf_counter()
    scaler             = StandardScaler()
    X_train_scaled     = scaler.fit_transform(X_raw.iloc[train_indices])
    X_val_scaled       = scaler.transform(X_raw.iloc[val_indices])
    preprocessing_time = float(time.perf_counter() - t_preprocess_start)
    n_features         = X_train_scaled.shape[1]
    y_train            = y[train_indices]; y_val = y[val_indices]
    X_train_tensor     = torch.from_numpy(X_train_scaled.astype(np.float32))
    X_val_tensor       = torch.from_numpy(X_val_scaled.astype(np.float32))
    y_train_tensor     = torch.from_numpy(y_train)
    y_val_tensor       = torch.from_numpy(y_val)
    model_registry     = build_model_registry(n_features)
    print(f'Fold {fold_idx + 1}/{N_SPLITS}  n_features={n_features}  train={len(train_indices)}  val={len(val_indices)}  preprocess={preprocessing_time:.4f}s')
    for model_name, model_factory, train_function, training_mode in model_registry:
        set_all_seeds(GLOBAL_SEED + fold_idx)
        if training_mode == 'sklearn':
            fold_result = train_sklearn_classifier(model_factory, X_train_scaled, y_train, X_val_scaled, y_val)
        else:
            fold_result = train_function(model_factory(), X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor)
        fold_result['preprocessing_time'] = preprocessing_time
        if model_name not in results_all['models']: results_all['models'][model_name] = {'folds': []}
        results_all['models'][model_name]['folds'].append(fold_result)
        print(f"  {model_name:42s} acc={fold_result['accuracy']:.4f} sens={fold_result['sensitivity']:.4f} spec={fold_result['specificity']:.4f} bal={fold_result['balanced_accuracy']:.4f} f1={fold_result['f1']:.4f} roc={fold_result['roc_auc']:.4f} pr={fold_result['pr_auc']:.4f} ll={fold_result['log_loss']:.4f} rmse={fold_result['rmse']:.4f} tr={fold_result['train_time']:.4f}s inf={fold_result['inference_time']:.6f}s")

Fold 1/10  n_features=30  train=512  val=57  preprocess=0.0075s
  ELM-linear-h50                             acc=0.9298 sens=1.0000 spec=0.8857 bal=0.9429 f1=0.9306 roc=1.0000 pr=1.0000 ll=0.4380 rmse=0.3612 tr=0.0087s inf=0.000453s
  ELM-linear-h100                            acc=0.9298 sens=1.0000 spec=0.8857 bal=0.9429 f1=0.9306 roc=1.0000 pr=1.0000 ll=0.4380 rmse=0.3612 tr=0.0032s inf=0.000185s
  ELM-linear-h200                            acc=0.9298 sens=1.0000 spec=0.8857 bal=0.9429 f1=0.9306 roc=1.0000 pr=1.0000 ll=0.4380 rmse=0.3612 tr=0.0060s inf=0.000323s
  ELM-sigmoid-h50                            acc=0.9825 sens=1.0000 spec=0.9714 bal=0.9857 f1=0.9825 roc=1.0000 pr=1.0000 ll=0.3716 rmse=0.3146 tr=0.0024s inf=0.000216s
  ELM-sigmoid-h100                           acc=0.9825 sens=1.0000 spec=0.9714 bal=0.9857 f1=0.9825 roc=1.0000 pr=1.0000 ll=0.3571 rmse=0.3035 tr=0.0040s inf=0.000217s
  ELM-sigmoid-h200                           acc=0.9474 sens=0.9091 spec=0.9714 bal=0.9403 

In [6]:
METRIC_KEYS = [
    'accuracy', 'precision', 'recall', 'f1',
    'sensitivity', 'specificity', 'balanced_accuracy',
    'roc_auc', 'pr_auc', 'log_loss', 'rmse',
    'train_time', 'time_per_epoch', 'inference_time', 'preprocessing_time', 'memory_mb'
]

def compute_fold_statistics(fold_values, confidence=0.95):
    arr = np.array(fold_values, dtype=float); finite_arr = arr[np.isfinite(arr)]
    if len(finite_arr) == 0:
        return {'mean': float('nan'), 'std': float('nan'), 'se': float('nan'),
                'ci_low': float('nan'), 'ci_high': float('nan'), 'values': arr.tolist()}
    n = len(finite_arr); mean_val = float(np.mean(finite_arr)); std_val = float(np.std(finite_arr, ddof=1))
    se_val = float(scipy.stats.sem(finite_arr)); t_critical = float(scipy.stats.t.ppf((1 + confidence) / 2.0, df=n - 1))
    margin = se_val * t_critical
    return {'mean': mean_val, 'std': std_val, 'se': se_val,
            'ci_low': mean_val - margin, 'ci_high': mean_val + margin, 'values': arr.tolist()}

summary_statistics = {}
for model_name, model_data in results_all['models'].items():
    fold_results = model_data['folds']; per_metric_stats = {}
    for metric_key in METRIC_KEYS:
        per_metric_stats[metric_key] = compute_fold_statistics([fold[metric_key] for fold in fold_results])
    summary_statistics[model_name] = per_metric_stats
results_all['summary'] = summary_statistics

print(f"{'Model':44s} {'Acc mean':>10} {'95% CI':>22} {'ROC mean':>10} {'F1 mean':>10} {'Sens':>8} {'Spec':>8}")
print('-' * 115)
for model_name, stats in summary_statistics.items():
    acc = stats['accuracy']; roc = stats['roc_auc']; f1 = stats['f1']
    sens = stats['sensitivity']; spec = stats['specificity']
    print(f"{model_name:44s} {acc['mean']:10.4f} [{acc['ci_low']:.4f}, {acc['ci_high']:.4f}]  {roc['mean']:10.4f} {f1['mean']:10.4f} {sens['mean']:8.4f} {spec['mean']:8.4f}")

Model                                          Acc mean                 95% CI   ROC mean    F1 mean     Sens     Spec
-------------------------------------------------------------------------------------------------------------------
ELM-linear-h50                                   0.9544 [0.9321, 0.9767]      0.9930     0.9545   0.9574   0.9523
ELM-linear-h100                                  0.9544 [0.9321, 0.9767]      0.9930     0.9545   0.9574   0.9523
ELM-linear-h200                                  0.9544 [0.9321, 0.9767]      0.9930     0.9545   0.9574   0.9523
ELM-sigmoid-h50                                  0.9684 [0.9530, 0.9838]      0.9911     0.9683   0.9526   0.9777
ELM-sigmoid-h100                                 0.9666 [0.9557, 0.9776]      0.9944     0.9665   0.9478   0.9776
ELM-sigmoid-h200                                 0.9667 [0.9485, 0.9849]      0.9877     0.9664   0.9387   0.9832
ELM-relu-h50                                     0.9544 [0.9241, 0.9847]      0.9

In [7]:
def make_serializable(obj):
    if isinstance(obj, dict): return {k: make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list): return [make_serializable(item) for item in obj]
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray): return make_serializable(obj.tolist())
    if isinstance(obj, np.bool_): return bool(obj)
    if isinstance(obj, float) and np.isnan(obj): return None
    return obj

serializable_results = make_serializable(results_all)
with open(OUTPUT_JSON, 'w', encoding='utf-8') as json_file:
    json.dump(serializable_results, json_file, indent=2, ensure_ascii=False)

print(f'Results saved to : {OUTPUT_JSON}')
print(f'Models evaluated : {len(results_all["models"])}')
print(f'Folds completed  : {N_SPLITS}')
print(f'Metrics per fold : {len(METRIC_KEYS)} + confusion_matrix')

Results saved to : results_BreastCancer.json
Models evaluated : 75
Folds completed  : 10
Metrics per fold : 16 + confusion_matrix
